In [3]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Update these file paths to your actual CSV file locations
# Examples of common file paths:
# For Windows: r"C:\Users\YourUsername\Documents\fake_news.csv"
# For Mac/Linux: "/Users/YourUsername/Documents/fake_news.csv"
# For current directory: "fake_news.csv" and "real_news.csv"

fake_path = "fake_news.csv"  # Change this to your actual fake news CSV file path
real_path = "real_news.csv"  # Change this to your actual real news CSV file path

# Alternative: Create sample data if files don't exist
def create_sample_data():
    """Create sample data for demonstration purposes"""
    fake_data = {
        'text': [
            'Breaking: Aliens land in New York City!',
            'Scientists discover chocolate cures all diseases',
            'Government hiding secret moon base',
            'Miracle diet makes you lose 50 pounds overnight',
            'Celebrity spotted with three-headed dog'
        ]
    }
    
    real_data = {
        'text': [
            'Stock market closes higher after economic report',
            'New research shows benefits of regular exercise',
            'Local school receives funding for new programs',
            'Weather forecast predicts rain this weekend',
            'City council approves new infrastructure project'
        ]
    }
    
    fake_df = pd.DataFrame(fake_data)
    real_df = pd.DataFrame(real_data)
    
    return fake_df, real_df

try:
    # Check if files exist before trying to read them
    if not os.path.exists(fake_path):
        print(f"Fake news file not found: {fake_path}")
        print("Creating sample data for demonstration...")
        fake, real = create_sample_data()
    elif not os.path.exists(real_path):
        print(f"Real news file not found: {real_path}")
        print("Creating sample data for demonstration...")
        fake, real = create_sample_data()
    else:
        # Load the CSV files if they exist
        fake = pd.read_csv(fake_path)
        real = pd.read_csv(real_path)
    
    print(f"Loaded {len(fake)} fake news articles and {len(real)} real news articles")
    
    # Add labels
    fake["label"] = 0  # Fake news
    real["label"] = 1  # Real news

    # Combine and shuffle data
    data = pd.concat([fake, real], ignore_index=True)
    data = data.sample(frac=1, random_state=42).reset_index(drop=True)

    # Select relevant columns (assuming 'text' column exists)
    if 'text' not in data.columns:
        print("Available columns:", data.columns.tolist())
        print("Please ensure your CSV files have a 'text' column")
    else:
        data = data[["text", "label"]].dropna()  # Remove any missing values

        X = data["text"]
        y = data["label"]

        # Split data with fixed random_state for reproducibility
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )

        # Vectorize text data
        vectorizer = TfidfVectorizer(
            stop_words="english", 
            max_features=10000,  # Limit features for better performance
            ngram_range=(1, 2)   # Include bigrams for better context
        )

        X_train_vec = vectorizer.fit_transform(X_train)
        X_test_vec = vectorizer.transform(X_test)

        # Train model
        model = LogisticRegression(random_state=42, max_iter=1000)
        model.fit(X_train_vec, y_train)

        # Evaluate model
        train_accuracy = model.score(X_train_vec, y_train)
        test_accuracy = model.score(X_test_vec, y_test)

        print(f"\nModel Performance:")
        print(f"Training Accuracy: {train_accuracy:.4f}")
        print(f"Testing Accuracy: {test_accuracy:.4f}")

        # Detailed evaluation
        y_pred = model.predict(X_test_vec)
        print(f"\nClassification Report:")
        print(classification_report(y_test, y_pred, target_names=['Fake', 'Real']))

except Exception as e:
    print(f"An error occurred: {e}")
    print("Please check your file paths and ensure the CSV files exist")

Fake news file not found: fake_news.csv
Creating sample data for demonstration...
Loaded 5 fake news articles and 5 real news articles

Model Performance:
Training Accuracy: 1.0000
Testing Accuracy: 1.0000

Classification Report:
              precision    recall  f1-score   support

        Fake       1.00      1.00      1.00         1
        Real       1.00      1.00      1.00         1

    accuracy                           1.00         2
   macro avg       1.00      1.00      1.00         2
weighted avg       1.00      1.00      1.00         2

